In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_silver_Folders"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Folders_Inventory" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_Folders" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 3, Finished, Available, Finished)

🔧 Initializing ntk_silver_Folders...
🚀 Starting ntk_silver_Folders


In [2]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 4, Finished, Available, Finished)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lit, current_timestamp, to_timestamp, row_number, coalesce, when,
    regexp_replace, upper, lower, isnan, isnull, concat_ws, desc
)
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql.window import Window

# Step 1: Initialize Spark Session with optimized configurations
spark = SparkSession.builder.appName("BronzeToSilver_ListsLibrary").getOrCreate()

## path
today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 
bronze_path = f"{source_path}/{year}/{month}/{day}/Folders_Inventory.csv"

# Step 2: Date-based path setup
current_date = datetime.now()
year = str(current_date.year)
month = f"{current_date.month:02d}"
day = f"{current_date.day:02d}"

silver_path = f"{target_path}/{year}/{month}/{day}/Dim_Folders.parquet"

# Step 3: Read CSV with schema inference

df_raw = spark.read.option("header", True).csv(bronze_path)
df_raw.show(2)


StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 5, Finished, Available, Finished)

+-----------+--------------------+------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+----------------+-----------+------------+---------+--------------+--------------+---------------+------------+-------------------+-------+------+-------------+--------+--------------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------------+----------------+-----------------+-------+----------------+--------------+-------------+------------------+-------------------+-------------+------------------+---------------+-----------------+--------------------+--------------+------------+------------+
|LibraryName|          LibraryUrl|  FolderName|          FolderPath|           FolderUrl|ItemId|            UniqueId|                GUID|              SiteId|     SiteName|             S

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, upper, regexp_replace, when, to_date, to_timestamp,
    row_number, monotonically_increasing_id, current_timestamp, lit
)
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window
from datetime import datetime

# --------------------------
# Step 4: Validations & Transformations
# --------------------------

df_cleaned = df_raw

# (1) Schema Normalization - trim whitespaces, lowercase col names
# df_cleaned = df_cleaned.toDF(*[c.strip().lower() for c in df_cleaned.columns])
# df_cleaned = df_cleaned.select([trim(col(c)).alias(c) for c in df_cleaned.columns])

# (2) UserGuid Logic - validate if it's a proper GUID, else NULL
df_cleaned = df_cleaned.withColumn(
    "GUID",
    when(col("GUID").rlike("^[0-9a-fA-F-]{36}$"), col("SiteId"))
    .otherwise(None)
)

# (3) ClaimsIdentifier Fix - prevent scientific notation, store as string
if "claimsidentifier" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn(
        "claimsidentifier",
        regexp_replace(col("claimsidentifier"), r"\.0$", "")  # remove decimal if present
    )

# (4) Standardize Boolean Columns (convert True/False, Yes/No, 1/0 → boolean)
bool_cols = [c for c in df_cleaned.columns if "is" in c or "flag" in c or "boolean" in c]
for c in bool_cols:
    df_cleaned = df_cleaned.withColumn(
        c,
        when(lower(col(c)).isin("true", "yes", "1"), True)
        .when(lower(col(c)).isin("false", "no", "0"), False)
        .otherwise(None)
    )

# (5) Date Normalization - standardize to yyyy-MM-dd
date_cols = [c for c in df_cleaned.columns if "date" in c or "created" in c or "modified" in c]
for c in date_cols:
    df_cleaned = df_cleaned.withColumn(
        c, to_date(col(c), "yyyy-MM-dd")
    )

# (6) Numeric Standardization - cast numeric-like columns
numeric_cols = [c for c in df_cleaned.columns if "count" in c or "id" in c or "number" in c]
for c in numeric_cols:
    df_cleaned = df_cleaned.withColumn(c, regexp_replace(col(c), "[^0-9]", ""))
    df_cleaned = df_cleaned.withColumn(c, col(c).cast(IntegerType()))

# (7) Filtering & Deduplication - drop null keys, remove duplicates
if "SiteId" in df_cleaned.columns:
    df_cleaned = df_cleaned.filter(col("SiteId").isNotNull())

window_spec = Window.partitionBy(df_cleaned.columns).orderBy(monotonically_increasing_id())
df_cleaned = df_cleaned.withColumn("row_num", row_number().over(window_spec))
df_cleaned = df_cleaned.filter(col("row_num") == 1).drop("row_num")

# (8) Data Validation Checks - add audit columns
df_cleaned = df_cleaned.withColumn("SnapshotDate", current_timestamp())
df_cleaned = df_cleaned.withColumn("record_valid", lit(True))  # placeholder validation flag



df_cleaned.show(2)


StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 6, Finished, Available, Finished)

+-----------+--------------------+----------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+----------------+-----------+------------+---------+--------------+--------------+---------------+------------+-------------------+-------+------+-------------+--------+--------------------+--------------------+-----------------+--------------------+-----------------+--------------------+-----------------+--------------------+--------------------+----------------+-----------------+-------+----------------+--------------+-------------+------------------+-------------------+-------------+------------------+---------------+-----------------+--------------------+--------------+------------+------+--------------------+------------+
|LibraryName|          LibraryUrl|FolderName|          FolderPath|           FolderUrl|ItemId|            UniqueId|                GUID|              Si

In [5]:

from pyspark.sql.functions import col, when, sha2, concat

# Creating FileKey hashvalue
df_cleaned = df_cleaned.withColumn(
    "FolderKey",
    when(
        col("FolderUrl").isNotNull(),
        sha2(col("FolderUrl"), 256)
    ).otherwise(None)
)

# Creating LibDocKey hashvalue
df_cleaned = df_cleaned.withColumn(
    "LibraryKey",
    when(
        col("LibraryUrl").isNotNull(),
        sha2(col("LibraryUrl"), 256)
    ).otherwise(None)
)

# Creating SiteKey hashvalue
df_cleaned = df_cleaned.withColumn(
    "SiteKey",
    when(
        col("SiteUrl").isNotNull(),
        sha2(col("SiteUrl"), 256)
    ).otherwise(None)
)

print(f"hash key for File / Library / Site created")

StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 7, Finished, Available, Finished)

hash key for File / Library / Site created


In [6]:
# --------------------------
# Step 5: Save to Parquet
# --------------------------
df_cleaned.write.mode("overwrite").parquet(silver_path)
# df_cleaned.printSchema()
print(f"✅ Data saved to {silver_path}")

StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 8, Finished, Available, Finished)

✅ Data saved to abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Folders.parquet


In [7]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = df_raw.count()   # Correct this to have a meaningful `rows_read`
    rows_written = df_cleaned.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 769c849f-12b6-4514-9069-4610cfc9bbec, 9, Finished, Available, Finished)

🔄 Starting ETL processing for ETL_Pipeline_Example...
Logging ETL Activity: {'Status': 'SUCCESS', 'StartTime': datetime.datetime(2025, 10, 15, 4, 57, 16, 814631), 'EndTime': datetime.datetime(2025, 10, 15, 4, 57, 17, 559005), 'DurationSeconds': 0.744374, 'RowsRead': 311, 'RowsWritten': 311, 'BytesProcessed': 524288000, 'ErrorDetails': None}
🎉 ETL_Pipeline_Example pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 311 → 311
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ETL_Pipeline_Example:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ETL_Pipeline_Example logging completed!
